# 遮罩語言模型（Masked Language Modeling）持續預訓練

## 學習目標

1. 理解 MLM 任務的原理：以 `[MASK]` token 替換輸入字元，訓練模型預測被遮罩的原始 token。
2. 掌握 2026 版 HuggingFace 訓練流程：`load_dataset` → `map(batched=True)` → `DataCollatorForLanguageModeling` → `Trainer`。
3. 了解 `DataCollatorForLanguageModeling` 如何動態執行隨機遮罩，避免手刻標籤邏輯。
4. 使用 `fill-mask` pipeline 驗證持續預訓練後的模型補全能力。

## 前置知識

- 需先閱讀 `02-Adv-tasks/01-text_classification/` 系列，了解 Trainer 基礎用法。
- MLM 是 BERT 系模型（BERT、RoBERTa、MacBERT）的核心預訓練目標；本 notebook 示範的是「持續預訓練（continued pre-training）」，即在特定領域語料上繼續訓練已有的 checkpoint。

## 銜接

- 上一個主題：`05-question_answering/` — 抽取式閱讀理解。
- 下一個主題：`07-causal_lm/` — 因果語言模型（GPT 風格）持續預訓練。

In [ ]:
# 版本鎖定：確保 2026 統一環境
# 在 Colab / 新環境執行時取消以下註解
# !pip install -q \
#     "transformers>=4.46" \
#     "datasets>=3.0" \
#     "accelerate>=1.0" \
#     "evaluate>=0.4" \
#     "safetensors>=0.4" \
#     "torch>=2.4"

## Step 1：匯入套件

2026 統一做法：
- `AutoModelForMaskedLM`：自動選取對應 masked LM head 的模型架構。
- `DataCollatorForLanguageModeling`：在 collate 階段動態對 batch 進行隨機遮罩（mlm_probability=0.15），無需預先生成帶 `-100` 標籤的資料集，大幅簡化資料處理。
- `Trainer` + `TrainingArguments`：取代手刻訓練迴圈，自動處理 gradient accumulation、mixed precision、logging。
- `set_seed`：固定亂數種子，確保可重現性。

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed,
)

set_seed(42)

## Step 2：載入資料集

優先使用 HuggingFace Hub 上的資料集 id，或透過環境變數 / `pathlib.Path` 指定本機快取路徑，避免硬路徑。

本 notebook 以 `wikimedia/wikipedia`（中文子集）做示範，實際持續預訓練時可替換成自有領域語料（格式相同即可）。

> **VRAM 注意：** 後續訓練以 `hfl/chinese-macbert-base`（~102M 參數）示範，bf16 下約佔 0.4 GB，加上 batch=16 的啟動記憶體共約 2–4 GB，一般消費級 GPU（8 GB）可執行。若資源更受限，可改用 `hfl/chinese-roberta-wwm-ext`（同規模）或直接跑 CPU（速度慢但功能完整）。

In [ ]:
import os
from pathlib import Path

# --- 資料來源設定 ---
# 方案 A（推薦）：從 HF Hub 串流下載，不需本機預先存放整份資料集
#   ds = load_dataset("wikimedia/wikipedia", "20231101.zh", split="train", streaming=True)
#
# 方案 B：本機路徑（透過環境變數，避免硬路徑）
#   DATA_DIR = Path(os.environ.get("WIKI_CN_DIR", "./wiki_cn_filtered"))
#   ds = load_dataset("parquet", data_dir=str(DATA_DIR), split="train")
#
# 以下使用小型公開語料做 end-to-end 示範（可替換成正式資料集）
ds = load_dataset("wikimedia/wikipedia", "20231101.zh", split="train")

print(ds)
print(ds[0].keys())

## Step 3：資料集前處理

### 3-1 初始化 Tokenizer

In [ ]:
MODEL_ID = "hfl/chinese-macbert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Vocab size:", tokenizer.vocab_size)
print("mask_token:", tokenizer.mask_token, "  id:", tokenizer.mask_token_id)

### 3-2 Tokenize 函式與 batched map

**為何用 `batched=True`？**

| 方式 | 速度 | 原因 |
|------|------|------|
| `map(fn)` 逐筆 | 1x | 每筆資料單獨呼叫 Python |
| `map(fn, batched=True)` | 3–5x | Tokenizer 底層以 Rust 批次處理 |

**`remove_columns`：** 原始欄位（`text`、`url`、`title`…）在 tokenize 後不再需要，移除可節省記憶體並確保 DataLoader 只看到 `input_ids`、`attention_mask` 等模型需要的欄位。

**`max_length=384`：** MacBERT 最大位置編碼為 512，取 384 保留適當邊界，同時兼顧 batch 記憶體效率。

In [ ]:
TEXT_COLUMN = "text"   # wikimedia/wikipedia 的文字欄位名稱
MAX_LENGTH = 384

def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        max_length=MAX_LENGTH,
        truncation=True,
        # 不在這裡 padding；動態 padding 交給 DataCollator
    )

tokenized_ds = ds.map(
    tokenize_fn,
    batched=True,
    num_proc=4,                     # 平行處理，加快速度
    remove_columns=ds.column_names, # 移除所有原始欄位
    desc="Tokenizing dataset",
)

print(tokenized_ds)
print(tokenized_ds[0].keys())

### 3-3 DataCollatorForLanguageModeling

這是 MLM 任務與一般分類任務在資料處理上最大的差異點：

- **動態遮罩（Dynamic Masking）**：每次 collate 時才隨機選取 15% token 替換為 `[MASK]`、隨機 token 或保留原值（比例 80/10/10），與 RoBERTa 做法一致。好處是同一筆資料在不同 epoch 會產生不同的遮罩組合，等同資料增強。
- **自動產生 `-100` 標籤**：未被遮罩的位置 label 設為 `-100`，讓 `CrossEntropyLoss` 自動忽略——這是框架替你處理的，不需手寫。
- **動態 Padding**：`DataCollatorForLanguageModeling` 底層呼叫 `pad` 到 batch 內最長序列，而非固定長度，節省約 20–40% 計算量（相較固定 512 padding）。

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,           # 啟用 Masked LM 模式（對應 BERT/MacBERT）
    mlm_probability=0.15,  # 遮罩 15% token，遵循 BERT 原始論文設定
    pad_to_multiple_of=8,  # 對齊 8 的倍數，讓 Tensor Core 效率最高
)

# 快速驗證 collator 行為
from torch.utils.data import DataLoader

_loader = DataLoader(tokenized_ds.select(range(4)), batch_size=2, collate_fn=data_collator)
_batch = next(iter(_loader))
print("input_ids shape:", _batch["input_ids"].shape)
print("labels shape:   ", _batch["labels"].shape)
print("Number of masked positions (non -100):",
      (_batch["labels"] != -100).sum().item())
del _loader, _batch

## Step 4：載入模型

2026 統一載入慣例：以 `torch_dtype=torch.bfloat16` 指定精度，搭配 `use_safetensors=True` 確保安全載入。

**為何選 bfloat16 而非 float16？**

| 精度 | 指數位元 | 尾數位元 | 動態範圍 | 訓練穩定性 |
|------|----------|----------|----------|-----------|
| fp32 | 8 | 23 | 最大 | 最穩 |
| **bf16** | **8** | **7** | **與 fp32 相同** | **接近 fp32** |
| fp16 | 5 | 10 | 小（易 overflow） | 需 loss scaling |

bf16 保留與 fp32 相同的指數位元，因此大梯度不會 overflow；A100/H100/RTX 3090+ 均原生支援。

> **`device_map='auto'` 注意：** MLM 訓練時不使用 `device_map='auto'`（model parallelism 與 Trainer 的 gradient checkpointing 有相容性限制）。`device_map='auto'` 主要用於大模型多卡推論或 QLoRA 場景（見 `04-PEFT` 模組）。此處改以 `TrainingArguments(bf16=True)` 讓 Trainer 負責混合精度控制。

In [ ]:
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)

# 參數量統計
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Step 5：配置訓練參數

**2026 完整 TrainingArguments 說明：**

- `bf16=True`：啟用 bf16 混合精度，相較 fp32 節省約 50% VRAM，訓練速度提升 1.5–2x。
- `optim='adamw_torch_fused'`：PyTorch 2.x 融合版 AdamW，比標準 AdamW 快 5–10%，記憶體相近。AdamW 相較 Adam 增加 weight decay 修正（Adam 的 weight decay 實作有數學錯誤，AdamW 才是正確版本）。
- `warmup_ratio=0.1`：前 10% 步驟線性 warmup，避免訓練初期梯度爆炸。
- `lr_scheduler_type='cosine'`：cosine annealing 讓學習率平滑衰減至 0，比 linear decay 通常收斂更好。
- `max_grad_norm=1.0`：梯度裁剪，防止偶發的大梯度步驟破壞已收斂的參數。
- `save_safetensors=True`：儲存時使用 safetensors 格式，安全且載入速度快。
- `eval_strategy='steps'`：每 N 步驗證一次，搭配 `load_best_model_at_end=True` 自動保留最佳 checkpoint。
- `seed=42`：確保 DataLoader shuffle 與 dropout 的可重現性。

**Effective batch size：**
```
effective_batch = per_device_train_batch_size × gradient_accumulation_steps × num_gpus
```
本設定在單 GPU 下：16 × 2 = 32，與 MacBERT 原始論文預訓練 batch 規模一致。

In [ ]:
# 訓練 / 驗證資料分割
# WikiMedia 資料集只有 train split；手動切出 1% 作為驗證集
split = tokenized_ds.train_test_split(test_size=0.01, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]

print(f"Train: {len(train_ds):,} samples  |  Eval: {len(eval_ds):,} samples")

In [ ]:
args = TrainingArguments(
    output_dir="./masked_lm",

    # --- 訓練超參 ---
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,    # effective batch = 16 * 2 = 32

    # --- 優化器 ---
    optim="adamw_torch_fused",        # PyTorch 2.x 融合版 AdamW（比標準快 ~5-10%）
    learning_rate=1e-4,               # 持續預訓練建議比 fine-tuning 大一個量級
    weight_decay=0.01,
    max_grad_norm=1.0,                # 梯度裁剪
    warmup_ratio=0.1,                 # 前 10% 步驟線性 warmup
    lr_scheduler_type="cosine",

    # --- 精度 ---
    bf16=True,                        # bf16 混合精度；需 Ampere+ GPU（A100/RTX 3090+）
    # fp16=True,                      # 若 GPU 不支援 bf16，改用此行（需 loss scaling）

    # --- 評估與儲存 ---
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,               # 只保留最近 2 個 checkpoint 節省磁碟
    save_safetensors=True,            # 儲存為 safetensors 格式
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # --- 日誌 ---
    logging_steps=50,
    report_to="none",                 # 設為 "wandb" 可啟用 W&B 追蹤

    # --- 可重現性 ---
    seed=42,
    data_seed=42,
)

print("Output dir:", args.output_dir)
print("Effective batch size:",
      args.per_device_train_batch_size * args.gradient_accumulation_steps)

## Step 6：建立 Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    # processing_class=tokenizer,  # transformers>=4.47 新增參數，用於取代舊版 tokenizer 參數
)

print("Trainer ready.")
print(f"Total training steps: ~{trainer.args.max_steps if trainer.args.max_steps > 0 else '(auto)'}")

## Step 7：模型訓練

> **VRAM 估算（`hfl/chinese-macbert-base`，bf16）：**
> - 模型權重：~0.4 GB
> - Adam optimizer states（2 份 fp32 copy）：~1.6 GB
> - batch=16 × seq=384 啟動記憶體：~1–2 GB
> - **合計約 3–5 GB**，8 GB GPU 可執行
>
> 若 VRAM 不足，可調低 `per_device_train_batch_size=8` 搭配 `gradient_accumulation_steps=4`，effective batch 不變。

In [ ]:
trainer.train()

## Step 8：儲存模型

2026 統一儲存慣例：`save_pretrained` 搭配 `safe_serialization=True`，生成 `.safetensors` 而非 `.bin`（pickle 格式）。

safetensors 優點：
1. **安全**：純資料格式，不執行任意 Python 程式碼，不存在 pickle 的 RCE 風險。
2. **速度**：記憶體映射（mmap）載入，比 pickle 快 2–10x。
3. **平行**：多線程讀取不同張量不需全部 deserialize。

In [ ]:
OUTPUT_DIR = "./masked_lm_final"

trainer.model.save_pretrained(OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model and tokenizer saved to: {OUTPUT_DIR}")

# 列出產生的檔案
import os
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f"  {f:40s}  {size_kb:8.1f} KB")

## Step 9：模型推理（fill-mask pipeline）

2026 統一 pipeline 載入慣例：使用 `device_map="auto"` 自動偵測可用 GPU；無 GPU 時自動 fallback 到 CPU，無需手動指定裝置索引。

此處直接載入剛訓練完的 in-memory model，無需從磁碟重新載入。

In [ ]:
from transformers import pipeline

# 使用 in-memory 的訓練後模型（避免重複載入磁碟）
# 注意：Trainer 若啟用 load_best_model_at_end=True，trainer.model 已是最佳 checkpoint
pipe = pipeline(
    "fill-mask",
    model=trainer.model,
    tokenizer=tokenizer,
    device_map="auto",   # 自動偵測 GPU，無 GPU 時 fallback 至 CPU
)

print("Pipeline ready.")

In [ ]:
# 範例 1：中文大學名稱補全
results = pipe(
    "西安交通[MASK][MASK]博物馆（Xi'an Jiaotong University Museum）是一座位于西安交通大学的博物馆"
)

print("Top predictions:")
for r in results:
    print(f"  score={r['score']:.4f}  token={r['token_str']!r}  sequence={r['sequence'][:40]}...")

In [ ]:
# 範例 2：新聞類型分類（遮罩位置應預測「科技」「財經」「娛樂」等）
results = pipe(
    "下面是一则[MASK][MASK]新闻。小编报道，近日，游戏产业发展的非常好！"
)

print("Top predictions:")
for r in results:
    print(f"  score={r['score']:.4f}  token={r['token_str']!r}  sequence={r['sequence'][:40]}...")

### （選用）從磁碟重新載入並推理

驗證儲存格式正確性，同時示範從 checkpoint 載入的完整流程。

In [ ]:
# 從 safetensors checkpoint 重新載入
_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
_model = AutoModelForMaskedLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)

_pipe = pipeline(
    "fill-mask",
    model=_model,
    tokenizer=_tokenizer,
    device_map="auto",
)

_results = _pipe("今天的天气[MASK][MASK]，适合出门运动。")
print("Reload test predictions:")
for r in _results:
    print(f"  score={r['score']:.4f}  token={r['token_str']!r}")

### （選用）推送至 HuggingFace Hub

持續預訓練完成後可公開發布模型，供後續 fine-tuning 使用。

In [ ]:
# 需先執行 huggingface-cli login 或設定 HF_TOKEN 環境變數
# HUB_MODEL_ID = "your-username/chinese-macbert-base-wiki-continued"
#
# trainer.model.push_to_hub(
#     HUB_MODEL_ID,
#     commit_message="Continued pre-training on wikimedia/wikipedia zh",
# )
# tokenizer.push_to_hub(HUB_MODEL_ID)
#
# print(f"Model pushed to: https://huggingface.co/{HUB_MODEL_ID}")

## 小結

本 notebook 示範了 2026 版 MLM 持續預訓練的完整流程，涵蓋以下關鍵環節：

- **資料載入**：透過 `load_dataset(hub_id)` 或環境變數指定本機路徑，確保跨環境可重現性。
- **Tokenize**：`map(batched=True, num_proc=4)` 利用 Tokenizer 的 Rust 後端批次處理，速度提升 3–5x。
- **動態遮罩**：`DataCollatorForLanguageModeling` 在 collate 階段執行隨機遮罩，並自動產生 `-100` 標籤，同時提供動態 Padding 節省計算量。
- **模型載入**：`torch_dtype=torch.bfloat16` + `use_safetensors=True`，兼顧記憶體效率與載入安全性。
- **優化器**：`adamw_torch_fused` 搭配 `warmup_ratio`、cosine scheduler、梯度裁剪，構成穩健的訓練配置。
- **儲存格式**：`safe_serialization=True` 生成 `.safetensors`，安全且載入速度快。
- **Pipeline 推理**：`device_map="auto"` 自動偵測裝置，無需手動管理 GPU index。

**關鍵觀念：**
- `DataCollatorForLanguageModeling` 的動態遮罩相當於一種資料增強——相同文本在不同 epoch 遮罩位置不同。
- 持續預訓練的學習率（1e-4）應大於 fine-tuning（2e-5 ~ 5e-5），讓模型真正吸收新領域知識。
- `eval_loss`（cross-entropy over masked positions）對應 perplexity：`ppl = exp(eval_loss)`，可用來比較不同 checkpoint 的語言建模能力。

## 練習題

1. 將 `mlm_probability` 改為 0.2，觀察訓練 loss 與 fill-mask 品質的變化。理論上為什麼更高的遮罩率可能反而讓任務更難學習？
2. 將訓練語料換成特定領域（如醫療、法律），重新訓練後比較 fill-mask 的預測分佈，解釋為什麼持續預訓練有助於後續的 fine-tuning。
3. 計算訓練前後的 perplexity（`math.exp(eval_loss)`），量化持續預訓練帶來的領域適應提升幅度。
4. 參考 `07-causal_lm/` notebook，比較 MLM（雙向注意力）與 CLM（單向注意力）在 fill-mask 任務上的本質差異。